In [148]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display, Markdown
import numpy as np
import sqlite3, json
import pandas as pd
import duckdb
import time
from typing import Literal, Optional
from pydantic import BaseModel, Field

In [ ]:
min_lon, max_lon = 125.678, 131.229
max_lat = 36.001

ais_df_filtered = test_result[
    (test_result['longitude'] >= min_lon) & (test_result['longitude'] <= max_lon) & (test_result['latitude'] <= max_lat)
]

ais_df_filtered.shape

In [149]:
local_llm = ChatOpenAI(
            api_key="ai",
            model="openai/gpt-oss-20b",
            base_url="http://192.168.0.110:8000/v1",
        )

In [ ]:
def compress_ship_data_duckdb(db_path, table_name, min_lon=125.678, max_lon=131.229, max_lat=36.001):
    db_path = db_path.replace("\\", "/") 
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL sqlite; LOAD sqlite;")
    con.execute(f"ATTACH '{db_path}' AS sqlite_db (TYPE SQLITE);")

    query = f"""
    WITH raw_data AS (
        -- 1. 데이터 로드 및 초기 필터링
        SELECT 
            *,
            CAST(timestamp AS TIMESTAMP) as ts,
            CAST(longitude AS DOUBLE) as lon_val,
            CAST(latitude AS DOUBLE) as lat_val,
            CAST(course AS DOUBLE) as c_course,
            CAST(speed AS DOUBLE) as s_speed,
            row_number() OVER () as temp_row_idx
        FROM sqlite_db.{table_name}
        WHERE longitude >= {min_lon} 
          AND longitude <= {max_lon} 
          AND latitude <= {max_lat}
    ),
    ordered_data AS (
        -- 2. 이전 값 계산 (정렬 안정성 확보)
        SELECT *,
            LAG(lon_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lon,
            LAG(lat_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lat,
            LAG(c_course) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_course,
            LAG(s_speed) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_speed
        FROM raw_data
    ),
    diff_calc AS (
        -- 3. 변화량 계산 (부동소수점 오차 방지를 위해 정수로 변환하여 비교)
        SELECT *,
            -- ROUND(3) 대신 1000을 곱해 정수로 만들어 비교 (Pandas와의 미세한 오차 제거)
            CASE WHEN CAST(lon_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lon, lon_val) * 1000 AS BIGINT) 
                   OR CAST(lat_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lat, lat_val) * 1000 AS BIGINT) 
                 THEN 1 ELSE 0 END as pos_change,
            
            CASE 
                WHEN (c_course - COALESCE(prev_course, c_course)) > 180 THEN (c_course - COALESCE(prev_course, c_course)) - 360
                WHEN (c_course - COALESCE(prev_course, c_course)) < -180 THEN (c_course - COALESCE(prev_course, c_course)) + 360
                ELSE (c_course - COALESCE(prev_course, c_course))
            END as course_diff,
            
            (s_speed - COALESCE(prev_speed, s_speed)) as speed_diff
        FROM ordered_data
    ),
    event_logic AS (
        -- 4. 이벤트 트리거 (>= 20 또는 >= 1.0 처럼 경계값에 미세 오차 고려 시도 가능하나 일단 정확히 일치 시도)
        SELECT *,
            CASE 
                WHEN pos_change = 1 
                  OR (ABS(course_diff) >= 20.0 AND s_speed >= 1.0)
                  OR ABS(speed_diff) >= 2.0 
                THEN 1 ELSE 0 
            END as event_trigger
        FROM diff_calc
    ),
    grouping AS (
        -- 5. 그룹 ID 생성 (Pandas cumsum과 동일)
        SELECT *,
            SUM(event_trigger) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id
        FROM event_logic
    ),
    summarized AS (
        -- 6. 그룹별 요약 (15개 컬럼 구성 시작)
        SELECT 
            ShipName,
            group_id,
            ANY_VALUE(mmsi) as mmsi, -- FIRST 대신 ANY_VALUE가 성능상 유리하나 의미는 같음
            ANY_VALUE(higher_types) as higher_types,
            ANY_VALUE(radius) as radius,
            MIN(ts) as start_time,
            MAX(ts) as end_time,
            ANY_VALUE(lon_val) as lon,
            ANY_VALUE(lat_val) as lat,
            ANY_VALUE(c_course) as first_course,
            AVG(s_speed) as avg_speed,
            ANY_VALUE(course_diff) as turn_val,
            ANY_VALUE(speed_diff) as accel_val
        FROM grouping
        GROUP BY ShipName, group_id
    ),
    final_stats AS (
        -- 7. 이전 그룹과의 속도 차이 계산
        SELECT *,
            avg_speed - COALESCE(LAG(avg_speed) OVER (PARTITION BY ShipName ORDER BY start_time), avg_speed) as group_speed_diff
        FROM summarized
    )
    -- 8. 최종 15개 컬럼 및 상태값 생성
    SELECT 
        ShipName, group_id, mmsi, higher_types, radius, start_time, end_time, 
        lon, lat, first_course, avg_speed, turn_val, accel_val, group_speed_diff,
        CASE 
            WHEN avg_speed < 1.0 THEN '정박/대기'
            ELSE 
                TRIM(
                    CONCAT_WS(' ',
                        CASE WHEN ABS(turn_val) >= 20.0 AND avg_speed >= 1.0 
                             THEN (CASE WHEN turn_val > 0 THEN '우선회' ELSE '좌선회' END) || '(' || ROUND(ABS(turn_val), 1) || '°)'
                             ELSE '' END,
                        CASE WHEN group_speed_diff >= 2.0 THEN '가속'
                             WHEN group_speed_diff <= -2.0 THEN '감속' -- < -2 대신 <= -2로 더 정확히
                             ELSE '' END,
                        CASE WHEN avg_speed >= 5.0 THEN '이동/통과' ELSE '저속 운항' END
                    )
                )
        END as status
    FROM final_stats
    ORDER BY ShipName, start_time
    """

    df_result = con.execute(query).df()
    con.execute("DETACH sqlite_db;")
    return df_result

In [7]:
start = time.time()
# testdb = "D:/AI_team/github/Vision_AI_RnD_team/projects/pj7(AIS_Simulator_Client/ships.db"
testdb = "D:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/test.db"
res = compress_ship_data_duckdb(testdb, "AIS_category")
print(time.time() - start)

NameError: name 'compress_ship_data_duckdb' is not defined

In [ ]:
res

In [8]:
def compress_ship_data_duckdb_further(db_path, table_name, min_time, max_time, min_lon=125.678, max_lon=131.229, max_lat=36.001):
    db_path = db_path.replace("\\", "/") 
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL sqlite; LOAD sqlite;")
    con.execute(f"ATTACH '{db_path}' AS sqlite_db (TYPE SQLITE);")

    query = f"""
    -- [STEP 1] Raw 데이터 로드 및 1차 트리거 (정수 변환 비교로 정밀도 확보)
    WITH raw_data AS (
        SELECT *,
            CAST(timestamp AS TIMESTAMP) as ts,
            CAST(longitude AS DOUBLE) as lon_val,
            CAST(latitude AS DOUBLE) as lat_val,
            CAST(course AS DOUBLE) as c_course,
            CAST(speed AS DOUBLE) as s_speed,
            row_number() OVER () as temp_row_idx
        FROM sqlite_db.{table_name}
        WHERE longitude >= {min_lon} AND longitude <= {max_lon} AND latitude <= {max_lat}
            AND timestamp BETWEEN '{min_time}' AND '{max_time}'
    ),
    ordered_data AS (
        SELECT *,
            LAG(lon_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lon,
            LAG(lat_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lat,
            LAG(c_course) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_course,
            LAG(s_speed) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_speed
        FROM raw_data
    ),
    diff_calc AS (
        SELECT *,
            CASE WHEN CAST(lon_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lon, lon_val) * 1000 AS BIGINT) 
                   OR CAST(lat_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lat, lat_val) * 1000 AS BIGINT) 
                 THEN 1 ELSE 0 END as pos_change,
            CASE 
                WHEN (c_course - COALESCE(prev_course, c_course)) > 180 THEN (c_course - COALESCE(prev_course, c_course)) - 360
                WHEN (c_course - COALESCE(prev_course, c_course)) < -180 THEN (c_course - COALESCE(prev_course, c_course)) + 360
                ELSE (c_course - COALESCE(prev_course, c_course))
            END as course_diff,
            (s_speed - COALESCE(prev_speed, s_speed)) as speed_diff
        FROM ordered_data
    ),
    event_logic AS (
        SELECT *,
            -- 경계값 오차 방지를 위해 0.000001 보정 (Pandas와의 일치성 향상)
            CASE WHEN pos_change = 1 OR (ABS(course_diff) >= 19.999999 AND s_speed >= 0.999999) OR ABS(speed_diff) >= 1.999999 THEN 1 ELSE 0 END as event_trigger
        FROM diff_calc
    ),
    grouping_v1 AS (
        SELECT *,
            SUM(event_trigger) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id_v1
        FROM event_logic
    ),
    summarized_v1 AS (
        -- [1차 압축] ANY_VALUE 대신 FIRST를 사용하여 Pandas .first()와 100% 일치화
        SELECT 
            ShipName, group_id_v1,
            FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
            MIN(ts) as start_time, MAX(ts) as end_time,
            FIRST(lon_val) as lon, FIRST(lat_val) as lat,
            FIRST(c_course) as first_course, AVG(s_speed) as avg_speed,
            FIRST(course_diff) as turn_val, FIRST(speed_diff) as accel_val
        FROM grouping_v1 
        GROUP BY ShipName, group_id_v1
    ),
    -- [STEP 2] 상태 판별 (부동소수점 오차 차단)
    status_calc AS (
        SELECT *,
            avg_speed - COALESCE(LAG(avg_speed) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), avg_speed) as group_speed_diff
        FROM summarized_v1
    ),
    status_final AS (
        SELECT *,
            CASE 
                -- 1.0, 20.0 등의 경계값을 소수점 8자리에서 반올림 후 비교하여 Pandas와 일치시킴
                WHEN ROUND(avg_speed, 8) < 1.0 THEN '정박/대기'
                ELSE TRIM(CONCAT_WS(' ',
                    CASE WHEN ROUND(ABS(turn_val), 8) >= 20.0 AND ROUND(avg_speed, 8) >= 1.0 
                         THEN (CASE WHEN turn_val > 0 THEN '우선회' ELSE '좌선회' END) || '(' || ROUND(ABS(turn_val), 1) || '°)' ELSE '' END,
                    CASE WHEN ROUND(group_speed_diff, 8) >= 2.0 THEN '가속' 
                         WHEN ROUND(group_speed_diff, 8) < -2.0 THEN '감속' ELSE '' END,
                    CASE WHEN ROUND(avg_speed, 8) >= 5.0 THEN '이동/통과' ELSE '저속 운항' END
                ))
            END as status
        FROM status_calc
    ),
    -- [STEP 3] 2차 압축
    v2_trigger AS (
        SELECT *,
            CASE WHEN status != COALESCE(LAG(status) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), status) 
                 THEN 1 ELSE 0 END as status_change
        FROM status_final
    ),
    v2_grouping AS (
        SELECT *,
            SUM(status_change) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id
        FROM v2_trigger
    )
    -- [STEP 4] 최종 요약 (집계 방식 일치)
    SELECT 
        ShipName, group_id,
        FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
        MIN(start_time) as start_time, MAX(end_time) as end_time,
        FIRST(lon) as lon, FIRST(lat) as lat,
        FIRST(first_course) as first_course,
        AVG(avg_speed) as avg_speed,
        FIRST(turn_val) as turn_val,
        FIRST(accel_val) as accel_val,
        FIRST(status) as status
    FROM v2_grouping
    GROUP BY ShipName, group_id
    ORDER BY ShipName, start_time
    """

    df_result = con.execute(query).df()
    con.execute("DETACH sqlite_db;")
    return df_result


In [139]:
start = time.time()
testdb = "D:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/test.db"
res_2 = compress_ship_data_duckdb_further(testdb, "AIS_category", '2022-12-01 00:00:00', '2022-12-01 23:59:59')
print(time.time() - start)

0.2417607307434082


In [140]:
res_2

,ShipName,group_id,mmsi,higher_types,radius,start_time,end_time,lon,lat,first_course,avg_speed,turn_val,accel_val,status
0,AGS- 신천지,0.0,352001086,others,5000,2022-12-01 00:00:14,2022-12-01 14:10:12,128.793800,35.077283,140.0,0.008718,0.0,0.0,정박/대기
1,AGS- 신천지,1.0,352001086,others,5000,2022-12-01 14:15:31,2022-12-01 14:15:31,128.791850,35.076550,248.0,2.600000,62.0,1.9,우선회(62.0°) 가속 저속 운항
2,AGS- 신천지,2.0,352001086,others,5000,2022-12-01 14:18:13,2022-12-01 14:18:13,128.791183,35.076317,246.0,3.300000,-2.0,0.7,저속 운항
3,AGS- 신천지,3.0,352001086,others,5000,2022-12-01 14:20:01,2022-12-01 14:24:12,128.786750,35.071650,205.0,7.200000,-41.0,3.9,좌선회(41.0°) 가속 이동/통과
4,AGS- 신천지,4.0,352001086,others,5000,2022-12-01 14:25:11,2022-12-01 14:25:11,128.781283,35.058417,195.0,11.200000,-10.0,4.0,가속 이동/통과
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
847,SS-079 홍범도,0.0,636021595,others,5000,2022-12-01 02:24:05,2022-12-01 02:59:52,125.695387,35.811957,13.3,16.120000,0.0,0.0,이동/통과
848,SSM-051,0.0,636093035,others,5000,2022-12-01 06:42:38,2022-12-01 07:12:58,129.358357,35.442553,334.0,0.020000,0.0,0.0,정박/대기
849,SSM-051,1.0,636093035,others,5000,2022-12-01 07:28:07,2022-12-01 07:28:07,129.363557,35.445630,8.1,4.800000,125.1,4.8,우선회(125.1°) 가속 저속 운항
850,SSM-051,2.0,636093035,others,5000,2022-12-01 07:43:08,2022-12-01 07:43:08,129.355618,35.453258,304.4,2.200000,-63.7,-2.6,좌선회(63.7°) 감속 저속 운항


In [59]:
def preprocess_ship_data(df):
    """
    선박 상태(status)를 기준으로 카테고리를 분류하고 
    필요한 컬럼(ShipName, mmsi)을 포함하여 전처리합니다.
    """
    
    # 1. 분류 조건 설정
    # 각 키워드가 포함되어 있는지 체크하는 조건 리스트
    conditions = [
        df['status'].str.contains("정박/대기", na=False),
        df['status'].str.contains("저속 운항", na=False),
        df['status'].str.contains("이동/통과", na=False) 
        # (참고: str.contains는 정규표현식 기반이라 매우 강력합니다)
    ]
    
    # 2. 각 조건에 매칭될 결과 값 (분류 명칭)
    choices = [
        "정박/대기",
        "저속운항",
        "운항/통과"
    ]
    
    # 3. 새로운 분류 컬럼 생성 (조건에 맞는 게 없으면 "기타"로 분류)
    df['category'] = np.select(conditions, choices, default="기타")
    
    # 4. 필요한 정보(ShipName, mmsi)가 포함된 결과 리스트 생성
    # 각 분류별로 데이터를 그룹화하여 리스트로 반환하거나 처리할 수 있습니다.
    report = {}
    for cat in choices:
        # 해당 카테고리에 속하는 배들만 필터링
        filtered_df = df[df['category'] == cat][['ShipName', 'mmsi', 'status']].drop_duplicates(subset='mmsi')
        report[cat] = filtered_df.to_dict('records')
        
    return df, report

In [65]:
df, report = preprocess_ship_data(res_2)

json_report = json.dumps(report, ensure_ascii=False, indent=4)

print(json_report)

{
    "정박/대기": [
        {
            "ShipName": "AGS- 신천지",
            "mmsi": 352001086,
            "status": "정박/대기"
        },
        {
            "ShipName": "AOE-59 화천",
            "mmsi": 431016257,
            "status": "정박/대기"
        },
        {
            "ShipName": "AST-688 일천봉",
            "mmsi": 256011000,
            "status": "정박/대기"
        },
        {
            "ShipName": "ATS- 평택",
            "mmsi": 215080000,
            "status": "정박/대기"
        },
        {
            "ShipName": "DDG-992 율곡이이",
            "mmsi": 431400833,
            "status": "정박/대기"
        },
        {
            "ShipName": "DDG-993 서애류성룡",
            "mmsi": 440314210,
            "status": "정박/대기"
        },
        {
            "ShipName": "DDH-971 광개토대왕",
            "mmsi": 305089000,
            "status": "정박/대기"
        },
        {
            "ShipName": "DDH-973 양만춘",
            "mmsi": 538008114,
            "status": "정박/대기"
        },
        {
         

In [177]:
class RouteQuery(BaseModel):
    datasource: Literal["ship_info", "none"] = Field(
        ...,
        description="사용자 질문에 따라 'ship_info' 또는 'none'으로 라우팅합니다."
    )
    mmsi: Optional[list[int]] = Field(
        default=None,
        description="""datasource가 'ship_info'일 때만 해당 선박의 MMSI 번호를 추출하여 포함합니다.
        mmsi가 2개 이상일 경우 mmsi를 모두 포함하고 따옴표나 쌍따옴표 없이 int형으로 출력합니다 'none'일 경우 이 필드는 비워둡니다(null)."""
    )

structured_llm_router = local_llm.with_structured_output(RouteQuery)

system = """당신은 사용자 질문을 '선박 정보(ship_info)' 또는 '기타(none)'로 분류하는 전문 라우터입니다.
1. ship_info 선택 기준:
   - 사용자의 질문에 선박 이름(Ship Name) 또는 MMSI 번호가 구체적으로 포함된 경우.
   - 선박의 현재 위치, 상태, 항적 등을 묻는 질문인 경우.

2. none 선택 기준:
   - 질문에 특정 선박 이름이나 MMSI 번호가 없는 경우.
   - 인사, 일반적인 대화, 또는 해운/선박과 관련 없는 질문인 경우.
   - 선박에 대해 묻고 있지만, '어떤 선박'인지 식별할 수 있는 정보(이름/MMSI)가 전혀 없는 경우.

출력 규칙:
- datasource가 'ship_info'인 경우, 질문에서 추출한 MMSI 번호를 함께 반환하십시오.
- datasource가 'none'인 경우, MMSI 필드는 비워두십시오."""

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human","{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

In [183]:
selected_data_source = question_router.invoke(
    {"question": "mmsi가 477016800, 305089000, 231123121, 244093231인 배가 현재 어디로 운항중이야"}
)

print(selected_data_source)

datasource='ship_info' mmsi=[477016800, 305089000, 231123121, 244093231]


In [152]:
default_chain = local_llm | StrOutputParser()

In [153]:
retriever_chains = {
    'ship_info': full_integrated_chain,
    'none': default_chain
}

In [184]:
def query_routing_report(question):
    full_text = ""
    handle = display(Markdown(""), display_id=True)
    
    selected_data_source = question_router.invoke({"question": question})

    total_chain = retriever_chains[selected_data_source.datasource]
    mmsi_list = selected_data_source.mmsi

    filted_mmsi = res_2_sorted[res_2_sorted['mmsi'].isin(mmsi_list)].copy()

    ship_list = [
        group[1].to_dict(orient='records') 
        for group in filted_mmsi.groupby('ShipName', sort=False)
    ]

    #mmsi 두개 이상일 때 각각의 최근위치
    last_df = filted_mmsi.groupby('mmsi').tail(1)
    coords = final_df[['latitude', 'longitude']].values
    weather_l = []
    
    for _, ship_row in last_df.iterrows():
        # 1. 선박 현재 위치 및 기본 정보 추출
        mmsi = ship_row['mmsi']
        curr_pos = np.array([ship_row['lat'], ship_row['lon']])
        
        # 2. 거리 계산 및 가장 가까운 지점 인덱스 추출
        dist_seq = np.sum((coords - curr_pos)**2, axis=1)
        closest_i = np.argmin(dist_seq)
        
        # 3. 가장 가까운 지점의 정보를 가져와서 선박 정보와 합치기
        closest_node = final_df.iloc[closest_i].to_dict()
        weather_l.append(closest_node)

    res = total_chain.invoke({
        "tracks": ship_list,
        "weathers": weather_l
    })
    
    return res
    

In [176]:
routing_rep = query_routing_report("mmsi가 477016800와 305089000인 배가 현재 어디로 운항중이야")
print(routing_rep)

**전반적 개요**

| 선박 | 주요 항해 흐름 | 특이 사항 | 관제 포인트 |
|------|----------------|-----------|-------------|
| **DDH‑971 광개토대왕** | - 00:00‑08:48 직선 주행 및 다수의 좌·우선회<br>- 08:19‑23:27 15시간 이상 정박 | - 03–05 시 지속적인 좌우선회(총 7회)로 교통 흐름에 지연<br>- 07:00‑07:55 가속/감속이 반복, 08:11 우선회 후 급속 감속<br>- 08:48 이후 정박으로 경로 변동이 중단 | - 3–5 시 연속 회전 시 선박 간 접촉 위험 감시<br>- 8–9 시 급속 회전/감속 시 인접선박 속도 차이 모니터링 |
| **DDH‑976 문무대왕** | - 00:00‑16:32 저속 운항/정박<br>- 16:32‑21:28 빠른 이동(≈11–12 kn)과 좌우선회 연속<br>- 22:24‑23:59 우선·좌선회가 계속되는 남동해지로 이동 | - 00:15‑16:26 정박 동안 잠재적 장비 점검 또는 훈련<br>- 16시 초반부터 가속·좌선회가 급격, 토콘 가능성<br>- 21:13‑21:28/22:24‑23:59 빠른 이동과 동일한 회전 패턴 | - 16시 초반 토콘 관찰 및 접촉 위험 최소화<br>- 21시 이후 대역 통행 중 선짐·선근 관리 |

---

### DDH‑971 광개토대왕

| 구간 | 주요 사건 | 속도 | 방향 |
|------|-----------|------|-------|
| **00:00‑03:00** | 26.5° 방향, 10.7 kn | 10.7 kn | 26.5° |
| **03:05** | 우선회(≈23°)→53° | 10.9 kn | 53° |
| **03:07‑05:20** | 53° 고정, 11.6 kn | 11.6 kn | 53° |
| **05:25** | 좌선회(≈28°)→331.6° | 11.4 kn | 331.6° |
| **05:30‑06:35** | 330° 안정, 12.2 

In [11]:
mmsi_counts = res_2['mmsi'].value_counts()
print(mmsi_counts[:20])

mmsi
305062000    163
440562000     50
352002106     38
372024000     37
563161700     36
431016257     33
441963000     33
215080000     31
371473000     28
538009816     25
352001086     24
374314000     21
477016800     19
440467000     19
636017515     19
431400833     18
466221000     17
477655100     17
256011000     16
219259000     15
Name: count, dtype: int64


In [13]:
# 1. 먼저 ShipName과 timestamp 기준으로 전체 데이터를 정렬합니다.
# (timestamp가 문자열이라면 정렬 전 pd.to_datetime으로 변환하는 것이 정확합니다)
res_2_sorted = res_2.sort_values(by=['ShipName', 'start_time'], ascending=True)

target_mmsi = [305062000, 440562000, 352002106, 372024000, 563161700, 431016257, 441963000, 215080000, 371473000, 538009816]
filtered_mmsi = res_2_sorted[res_2_sorted['mmsi'].isin(target_mmsi)].copy()


In [14]:
#2개 이상의 항적 각각 리스트로 묶음
ship_track_list = [
    group[1].to_dict(orient='records') 
    for group in filtered_mmsi.groupby('ShipName', sort=False)
]

trajectory_json = json.dumps(ship_track_list, ensure_ascii=False, indent=4, default=str)
print(trajectory_json)

[
    [
        {
            "ShipName": "AOE-59 화천",
            "group_id": 0.0,
            "mmsi": 431016257,
            "higher_types": "auxiliary",
            "radius": 5500,
            "start_time": "2022-12-01 06:06:12",
            "end_time": "2022-12-01 08:20:52",
            "lon": 129.95,
            "lat": 33.475,
            "first_course": 0.0,
            "avg_speed": 0.4,
            "turn_val": 0.0,
            "accel_val": 0.0,
            "status": "정박/대기"
        },
        {
            "ShipName": "AOE-59 화천",
            "group_id": 1.0,
            "mmsi": 431016257,
            "higher_types": "auxiliary",
            "radius": 5500,
            "start_time": "2022-12-01 08:32:19",
            "end_time": "2022-12-01 08:32:19",
            "lon": 129.9511933,
            "lat": 33.482015,
            "first_course": 322.6,
            "avg_speed": 2.6,
            "turn_val": -98.29999999999995,
            "accel_val": 1.8,
            "status": "좌선회(98.

In [16]:
#날씨DB에서 데이터 로드
conn = sqlite3.connect('d:/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/korea_weather.db', check_same_thread=False)
weather_cursor = conn.cursor()

# 4. 최종 DB 포맷 조립 (예: '2025-12-10 3')
current_hour_str = "2025-12-10 3"

query = """
    SELECT 
        b.지점명,
        b.latitude, 
        b.longitude, 
        w.*
    FROM 
        weather_buoy AS w
    JOIN 
        buoy_position AS b ON w.지점 = b.지점
    WHERE 
        w.일시 LIKE ?
"""

# 3. 쿼리 실행
search_param = f"{current_hour_str}:%"
weather_cursor.execute(query, (search_param,))
rows = weather_cursor.fetchall()

# 4. 데이터 출력 및 처리
if rows:
    # 2. Pandas 데이터프레임으로 로드
    col_names = [desc[0] for desc in weather_cursor.description]
    df = pd.DataFrame(rows, columns=col_names)

    target_cols = [0, 1, 2] + list(range(5, len(df.columns)))
    final_df = df.iloc[:, target_cols]


In [17]:
#mmsi 두개 이상일 때 각각의 최근위치
last_rows_df = filtered_mmsi.groupby('mmsi').tail(1)
ref_coords = final_df[['latitude', 'longitude']].values
weather_results = []

for _, ship_row in last_rows_df.iterrows():
    # 1. 선박 현재 위치 및 기본 정보 추출
    mmsi = ship_row['mmsi']
    current_pos = np.array([ship_row['lat'], ship_row['lon']])
    
    # 2. 거리 계산 및 가장 가까운 지점 인덱스 추출
    dist_sq = np.sum((ref_coords - current_pos)**2, axis=1)
    closest_idx = np.argmin(dist_sq)
    
    # 3. 가장 가까운 지점의 정보를 가져와서 선박 정보와 합치기
    closest_node_data = final_df.iloc[closest_idx].to_dict()
    weather_results.append(closest_node_data)

weather_json = json.dumps(weather_results, ensure_ascii=False, indent=4)
print(weather_json)

[
    {
        "지점명": "울산",
        "latitude": 35.3453,
        "longitude": 129.8414,
        "풍속(m/s)": 5.9,
        "풍향(deg)": 337.0,
        "GUST풍속(m/s)": 8.2,
        "현지기압(hPa)": 1026.5,
        "습도(%)": 53,
        "기온(°C)": 8.8,
        "수온(°C)": 19.0,
        "최대파고(m)": 1.5,
        "유의파고(m)": 0.9,
        "평균파고(m)": 0.6,
        "파주기(sec)": 7.1,
        "파향(deg)": 23
    },
    {
        "지점명": "울산",
        "latitude": 35.3453,
        "longitude": 129.8414,
        "풍속(m/s)": 5.9,
        "풍향(deg)": 337.0,
        "GUST풍속(m/s)": 8.2,
        "현지기압(hPa)": 1026.5,
        "습도(%)": 53,
        "기온(°C)": 8.8,
        "수온(°C)": 19.0,
        "최대파고(m)": 1.5,
        "유의파고(m)": 0.9,
        "평균파고(m)": 0.6,
        "파주기(sec)": 7.1,
        "파향(deg)": 23
    },
    {
        "지점명": "거제도",
        "latitude": 34.7667,
        "longitude": 128.9,
        "풍속(m/s)": 5.5,
        "풍향(deg)": 325.0,
        "GUST풍속(m/s)": 9.1,
        "현지기압(hPa)": 1027.4,
        "습도(%)": 55,
        "

In [9]:
#mmsi 한개일 때 최근위치
last_data = filtered_mmsi.iloc[-1]
last_lat = last_data['lat']
last_lon = last_data['lon']

print(last_lat, last_lon)

33.17238333 126.9334333


In [11]:
target = np.array([last_lat, last_lon])

dist_sq = np.sum((final_df[['latitude', 'longitude']].values - target)**2, axis=1)

closest_idx = np.argmin(dist_sq)

closest_location = final_df.iloc[closest_idx]
print(closest_idx)
closest_location['지점명']

12


'서귀포'

In [29]:
weather_dict = closest_location.to_dict()
weather_json = json.dumps(weather_dict, ensure_ascii=False, indent=4, default=str)

print(weather_json)

{
    "지점명": "서귀포",
    "latitude": 33.1281,
    "longitude": 127.0228,
    "풍속(m/s)": 7.6,
    "풍향(deg)": 7.0,
    "GUST풍속(m/s)": 9.9,
    "현지기압(hPa)": 1026.5,
    "습도(%)": 55,
    "기온(°C)": 12.2,
    "수온(°C)": 20.4,
    "최대파고(m)": 1.2,
    "유의파고(m)": 0.8,
    "평균파고(m)": 0.5,
    "파주기(sec)": 4.5,
    "파향(deg)": 1
}


In [21]:
#vllm server alive check
res = local_llm.invoke("오늘 날씨가 어때?")
res

AIMessage(content='죄송하지만 현재는 실시간 날씨 정보를 제공할 수 없습니다.  \n어떤 도시에 대한 날씨를 알고 싶으신가요? 알려주시면 그 지역의 최신 기상 정보를 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 225, 'prompt_tokens': 76, 'total_tokens': 301, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'id': 'chatcmpl-62b898325df646a6b7d6b15f29eda860', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d6090-890d-7793-aef3-206cc7e21ec9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 225, 'total_tokens': 301, 'input_token_details': {}, 'output_token_details': {}})

In [22]:
ship_track_list

[[{'ShipName': 'AOE-59 화천',
   'group_id': 0.0,
   'mmsi': 431016257,
   'higher_types': 'auxiliary',
   'radius': 5500,
   'start_time': Timestamp('2022-12-01 06:06:12'),
   'end_time': Timestamp('2022-12-01 08:20:52'),
   'lon': 129.95,
   'lat': 33.475,
   'first_course': 0.0,
   'avg_speed': 0.4,
   'turn_val': 0.0,
   'accel_val': 0.0,
   'status': '정박/대기'},
  {'ShipName': 'AOE-59 화천',
   'group_id': 1.0,
   'mmsi': 431016257,
   'higher_types': 'auxiliary',
   'radius': 5500,
   'start_time': Timestamp('2022-12-01 08:32:19'),
   'end_time': Timestamp('2022-12-01 08:32:19'),
   'lon': 129.9511933,
   'lat': 33.482015,
   'first_course': 322.6,
   'avg_speed': 2.6,
   'turn_val': -98.29999999999995,
   'accel_val': 1.8,
   'status': '좌선회(98.3°)  저속 운항'},
  {'ShipName': 'AOE-59 화천',
   'group_id': 2.0,
   'mmsi': 431016257,
   'higher_types': 'auxiliary',
   'radius': 5500,
   'start_time': Timestamp('2022-12-01 08:35:17'),
   'end_time': Timestamp('2022-12-01 14:50:08'),
   'lon': 

In [37]:
ais_prompt_template ="""
    # Role
    당신은 대한민국 주변 선박운행을 관제하는 베테랑 해상 관제사(VTS Operator)이자 선박 항적 분석 전문가 입니다. 
    제공된 요약된 항적 데이터(Summarized Trajectory)를 바탕으로 선박의 이동 패턴과 주요 이벤트를 전문적인 자연어로 묘사해주세요.
    
    #Input Data
    1. Summarized Trajectory
    {trajectory_data}
    2. Current Weather data
    {weather_data}
    
    # Guidelines
    1. 시간 순서대로 전체적인 항해 흐름을 요약하고 선박이름을 명시해주세요.
    2. 요약할 때 기준은 "status"필드를 기준으로 하되 start_time과 end_time을 참고하세요. 
    3. trajectory_data는 ShipName이 같은 데이터끼리 리스트로 한번 더 묶여있으니 서로 ShipName이 서로 다른 배끼리 혼용하지 않도록 유의하세요.
    4. 첫번째 ShipName에 해당하는 현재날씨는 weather_data에서 첫번째 날씨 데이터를, 두번째 Shipnam에 해당하는 현재 날씨는 weather_data에서 두번째 날씨 데이터를 참고하세요.
    5. 전문적인 해상 관제 용어를 사용해도 되지만, 가독성이 좋게 작성하세요.
    6. 날씨데이터에 대해서 관제사가 고려하고 참고해야 할 사항에 대해 설명해주세요
    
    #Output Format
    1. 전체적인 항해 패턴을 간단하게 요약; 관제사가 관심을 가져야 할 특이 사항이 있을 시 간단하게 언급
    2. 현재 날씨에 대해 간단히 요약; 관제사가 관심을 가져야 할 날씨의 특이사항(풍향, 풍속 유의파고 등)이 있을 시 간단하게 언급 및 조치사항 설명
    3. 1, 2 항목을 종합적으로 정리해서 묘사
    """

each_ship_prompt = ChatPromptTemplate.from_template(ais_prompt_template)

In [ ]:
ship_passthrough_chain = (
    RunnablePassthrough.assign(
        trajectory_data = lambda x: trajectory_json,
        weather_data = lambda x: weather_json,
    )
    | each_ship_prompt
    | local_llm
)

# each_ship_chain = each_ship_prompt | local_llm | StrOutputParser()

In [62]:
ans = ship_passthrough_chain.invoke({'question': "항적에 대해 묘사해줘"})
display(Markdown(ans.content))

**1. 전체 항해 흐름(시각 순) 및 관제상의 핵심 포인트**  

- **AOE‑59 화천**  
  - **06:06–08:20** : 정박/대기 → 8:32 “좌선회(98.3°)” 후 저속(2.6 kn) 정박/대기.  
  - **08:35–14:50** : 장시간 정박(정박/대기) – 항해 중단 (6 h 15 min).  
  - **14:55–15:07** : 20‑km 가시거리 내 가속해 “우선회(88.4°)” → 이동/통과(8 kn).  
  - **15:07–17:00** : 10‑12 kn 속도로 선박 이동, 주행 중 21:00‑18:10 도달.  
  - **18:10–18:20** : 끝에 48° 좌선회 후 11 kn·→ 10~12 kn → 18:18~18:20에 여러 번 방향 전환.  
  - **18:20–19:54** : 11.8 kn 속도로 계속 이동(통과). 19:54–20:12 : 12.3 kn 속도로 미세 주행.  
  - **20:15–20:41** : 주로 12~14 kn→13 kn 영역으로 이동, 20:20에서 26.2° 좌선회, 20:35에서 29° 우선회 등 빈번한 선회(운항 경로가 20~33°에서 81°까지 다양).  
  - **특이 사항** – 6 ~ 7 hr 가시거리 감소와 동시에 정박이 지속된 시점이 있으므로 선박이 정박/대기 모드 운항 중 위험요소(예: 기상 변화에 대한 빠른 대응)가 필요.  
  - **전략요소** – 엔진 가동 중단 시, 기류와 파고에 따라 부양/안전 조건이 바뀔 수 있으므로 상황별 자세한 모니터링이 요구.

- **FFG‑818 대구**  
  - **00:00–06:20** : 반복적 저속 운항/정박/대기(1–2 kn).  
  - **07:40–08:00** : 0.88 kn 정박 → 07:40~08:00 동안 항해 모드(저속).  
  - **08:05–11:00** : 1–1.5 kn 저속 운항, 08:05부터 11:00까지 경로라면 선택.  
  - **11:05–14:15** : 1.1 kn 선행, 3‑5 kn 보행, 13:00~14:15 동안 2.9 kn(우선회 90°) 협동시간.  
  - **14:21–15:30** : 급속(4.7 kn) 엔진 가동 후 14:35~14:41 5.3 kn 이동, 14:47~14:50 우선회 25–24°.  
  - **15:00–18:40** : 5–7 kn 범위 내 선박 운항, 15:05~15:06 가속 10.5 kn, 15:06~15:55 → 4.5 ~ 7 kn.  
  - **18:50–23:30** : 8.0 ~ 11.9 kn 속도차. 20:55-22:10 11 ~ 12 kn, 22:30 5 kn, 23:30 명확한 11.9 kn.  

  - **특이 사항** – 14:35에서 우선회(90°) 과 14:44 급속, 8 시간 간격 내에 감속-가속 대역이 빈번해 3 마리가 승무원 수와 연료 소모를 주시해야 함.  

- **LST‑683 향로봉**  
  - **00:00–02:00** : 12–15 kn 범위 내 운항, ‘가속(소수분)’ 부여, 방향 전환(우선회 40° ~ 35°, 좌선회 64°).  
  - **02:00–03:30** : 14–15 kn, 큰 선회(150°~130°) 포함.  
  - **03:30–04:30** : 13–15 kn, 2조·15h, 중점적 선회(16–70°) 가속→감속.  
  - **04:30–05:30** : 10–13 kn, 경량 운항, 3시~5시 사이에 급속(13.5 kn) 가속 후 4:50~5:00속도 13.1 kn, 5:05 14.07 kn.  
  - **05:00–06:00** : 12–15 kn, 5:55 가속 14 kn, 6:05 ~6:30 12–13 kn, 6:40 13 ~ 15 kn.  
  - **05:45–05:50** : 감속 6 kn 전환위험을 주시 필요.  

  - **특이 사항** – 육지 근접 시 부드러운 선회와 가속이 반복적으로 등장. 항만/해협 접근 때 선박이 자유계통을 띄겠습니다.

- **SS‑078 유관순**  
  - **04:03–08:03** : 긴 정박(0 kn), 시계 4 전후 0V.  
  - **08:11** : 14.4 kn•28.5° 우선회 → 08:24 19.0 kn→20.29 kn(좌선회 53.7°).  
  - **08:26–10:49** : 21 kn~21.5 kn 지속, 08:26 21.3 kn→10:49 21.8 kn.  
  - **11:04** : 20.9 kn(우선회 40.9°) – 같은 기간(11:04‑11:30) 19–22 kn 근중.  
  - **12:14–12:30** : 19.86‑20.27 kn(좌선회 26°~72°) – 강한 목표 보드 26.5°.  
  - **13:30–17:30** : 21.3–22.5 kn 지속, 좌우 선회 (82°) 등 역동적.  
  - **20:00 이후** : 15 kn∙~ 21.5 kn 운항, 20:04 4 kn 가속, 20:10 21.5 kn 등.  

  - **특이 사항** – 8:00 ~ 11:00 동안 22 kn 이상 직진, 12:14 ~ 12:25 중 115°/@115° 우선회, 12:14~12:56 사이 좌우 대화속 26° ~ 78° 급전개. 관제시는 선종과 항구가능 해기에 따라 동시 놓곳~~.

**2. 현재 날씨 요약 & 관제시 고려 사항**  

| 지역(해상먼)** | 풍향 | 풍속 | Gust | 최대파고 | 유의파고 | 평균파고 | 파주기 |
|---|---|---|---|---|---|---|---|
| 울산(1st) | 337° | 5.9 m/s | 8.2 m/s | 1.5 m | 0.9 m | 0.6 m | 7.1 s |
| 소매물도(2nd) | 9° | 4.5 m/s | 5.5 | 0.5 | 0.3 | 0.3 | 8.4 |
| 남해111(3rd) | 25° | 6.4 m/s | 8.7 | 0.9 | 0.6 | 0.4 | 4.0 |
| 신안(4th) | 54° | 3.6 m/s | 3.6 | 0.2 | 0.1 | 0.1 | 5.7 |

- **울산**: 대기압 ~1026 hPa, 주비 및 고기압, 풍속 5.9 m/s(≈11 kn) 시령이 동쪽에서 북쪽으로, 유난히 높은 첫 번째 파고(1.5 m)와 초고기동(0.9 m) 표시. 관제자는 **AOE‑59**의 상대적 좌우선회 움직임이 풍향과 비례하고, 정박대기 장기 운항 시 **백미가** 고압 유동을 주의해야 함. 특히 선박가속 8 kn이 풍저항과 조합될 때 부조성 부양량이 끌린다.  

- **소매물도**: 부드러운 4 m/s 풍속, 상온, 파고 0.5 m. 선군 관제는 선박의 정박/저속 운항에 대해 큰 위험 부피를 두지 않아도 됨.  

- **남해111**: 풍속이 6.4 m/s, 파고 0.9 m; 선양에 비례해 **LST‑683**와 **SS‑078**가 15‑22 kn 운항 시 해당 파동이 부하에 부가·가속의 부담을 주는 데 주목. 빠른 파주기가 4 s·에 방폭추세, 갑작스런 파고 상승에 대비해 SOP(기술원 관리팀, 선박 선회/가속/감속)기준을 순응.  

- **신안**: 3.6 m/s 풍속, 경미한 파고(0.2 m). **SS‑078**에 해당 지역이 아웃사이딩으로 위치해 고성능까지 요구, 근근 최악 고요해건이 될 수 자. 

**3. 종합 평가 및 조치 항목**  

1. **AOE‑59 화천**은 긴 정박(6‑7 h) 후 이동에 대한 환경 불안정성을 보이며, **울산의 0.9 m** 유의파고와 5.9 m/s 풍향의 변동이 펼쳐질 가능성이 있다. 고정고정 보급과 일정 파라미터(엔진 가동·정지) 관리가 필요.  

2. **FFG‑818 대구**는 ‘저속/정박’과 ‘10‑12 kn 고속’이 섞이면서 135°~90° 급우선회가 대중적이다. 소매물도의 풍향(9°)이 대체로 동쪽에서, 고속 선주진 시 풍저항이 낮으므로 빠른 속도 재시작에 유리하지만 급 가속·감속이 연료소비를 높이므로 운항 계획을 신중히 설계해야 한다.  

3. **LST‑683 향로봉**은 12–15 kn 범위 내 이동과 20° 재주진·가속이 혼재돼 해역의 강풍(25°)과 1 m 이하의 파고에 의해 부양이 세차질될 수 있다. 대진선형 운항 시 선박 정지(정박) + 부양기준(전기 0.9 m) 체크이 중요하다.  

4. **SS‑078 유관순**은 21–22 kn 고속 운항이 반복되며, 신안 지역에서 3.6 m/s 풍향(54°)이무방향으로이나 급격한 기류 변동 가능성(고기압 -> 저기압) 진입 시 뛔에 주의를 기울여야 한다.  

**관제 조치**  
- **정박·대기** 리스팅되는 배(특히 AOE‑59, FFG‑818)은 조기 시청과 귀가 경로 예측을 위해 **경정요 (Latitude/Longitude), 현중 (arruam**) 운용이 필수.  

- **고속 이동** 배(SS‑078, LST‑683)는 **파고·풍향**가 자체 모드에 미치는 구조적 영향을 파악할 수 있는 **RMS(GTUR)** 시스템 보고/컨트롤을 필요.  

- **가속/감속**을 반복하는 배(FFG‑818, AOE‑59)는 바디파워(레버리지)와 대비해 **FuelConsumption**, 엔진볼륨을 재조정하기 위한 **NCU(제어권)* 조율이 필요.  

- **날씨경보** 방지로 각 배와 관제간 30분 간격 **MFR** 정보를 공유하며, **관제 트랜스파러시성** 강화가 요구.  

**최종상황**  
- 각 배는 미리 계획된 항해경로를 대체로 따랐으나 파동과 풍향 변동으로 인해 선박가속·감속, 선회이 점진적으로 표현되었다.  
- 날씨 데이터의 **풍향·풍속** 및 **파고**는 대부분 **저(≤1 m)**이지만, **울산** 지역의 1.5 m 파고는 ‘정박/대기’ 시 위험을 증가시킨다.  
-መ\n 관제가 현재 일정 대비, 부력을 고려한 **소음/연료** 모니터링을 강화해 **선박 안전** 및 **환경 규제**를 동시에 달성하기에 최적의 상황이다.

In [ ]:
astream_list = []
async def stream_output():
    # astream을 사용하여 비동기 스트리밍
    async for chunk in ship_passthrough_chain.astream({'question': "항적에 대해 묘사해줘"}):
        # 주피터 노트북에서 바로 출력 (end=""로 줄바꿈 없이 출력)
        print(chunk.content, end="", flush=True)
        astream_list.append(chunk.content)

# 주피터에서 비동기 함수 실행
await stream_output()

In [ ]:
def run(tr_list):
    formatted_data = json.dumps(tr_list, ensure_ascii=False, indent=4, default=str)
    return each_ship_chain.invoke({"trajectory_data": formatted_data})
        
output = []
with ThreadPoolExecutor(max_workers=100) as executor:
    # total_list의 각 sub_list를 run에 하나씩 매핑
    results = list(executor.map(run, total_list))

output.extend(results)

In [51]:
summary_trajectory_template ="""
    #Input Data
    {all_analyses}
    
    # Guidelines
    1. Input Data의 정보를 종합해서 전체 배에 대한 내용을 종합적으로 요약해주세요
    2. 요약 시 각각의 배의 이름을 명시하고 특이사항을 중점적으로 요약하세요 
    3. 수치는 바뀌지 않도록 정확하게 참고해주세요
    4. 반복적인 내용은 짧게 요약하세요
    """

summary_trajectory_prompt = ChatPromptTemplate.from_template(summary_trajectory_template)

In [26]:
#for문을 이용해 list를 만든 다음 parallel로 병렬처리 하는 로직
analyze_chain = each_ship_prompt | local_llm | StrOutputParser()

parallel_steps = {}
for i in range(len(ship_track_list)):
    # 각 실행 경로(ship_0, ship_1...)가 고유의 데이터를 갖도록 클로저(i=i) 사용
    parallel_steps[f"ship_{i}"] = (
        RunnableLambda(lambda x, i=i: {
            "trajectory_data": ship_track_list[i],
            "weather_data": results[i]
        }) 
        | analyze_chain
    )

parallel_analyses = RunnableParallel(parallel_steps)

# 3. 모든 분석 결과를 하나로 합치는 함수
def combine_all_analyses(analyses_dict):
    combined_text = ""
    for key, text in analyses_dict.items():
        combined_text += f"\n[{key} 분석 결과]\n{text}\n"
    return {"all_analyses": combined_text}

summary_chain = summary_trajectory_prompt | local_llm | StrOutputParser()

# 5. 전체 파이프라인 연결
# Parallel 결과 -> Lambda(합치기) -> Summary(최종 요약)
full_pipeline = (
    parallel_analyses 
    | RunnableLambda(combine_all_analyses)
    | RunnableLambda(lambda x: {**x, "ship_count": len(ship_track_list)}) # 선박 수 추가
    | summary_chain
)

In [27]:
# 6. 실행 (입력값이 parallel 내부에서 생성되므로 빈 딕셔너리로 시작 가능)
start = time.time()
final_result = full_pipeline.invoke({})
print(time.time() - start)

107.77638578414917


In [28]:
display(Markdown(final_result))

## 1. 전량 배들의 종합 정리  
| 항목 | 내용 |
|------|------|
| **운항 범위** | 2022‑12‑01 00 :00 – 23 :59, 울산·거제·남해·북해 인근 해역 |
| **주요 운항 패턴** | * 정박·대기(≤ 1 kn) : 3–4 h 정도 지속되는 구간이 다수; <br>* 고속 주행(≈ 10–15 kn) : 대부분 1–2 h 지속 |  
| **선회** | 20–80 ° 범위에서 급격 전환이 반복되고, 100 ~ 150 ° 이상의 큰 전환을 한 배도 존재 |  
| **날씨** |  | - 풍향 337° (남서‑서쪽) | - 풍속 5.9 m / s (~10.5 kn) | - GUST 8.2 ~ 9.1 m / s (≈ 14–19 kn) | - 평균 파고 0.4 ~ 0.6 m, 최대 1.5 m | - 기온 8.8 ~ 11.6 °C, 수온 19 ~ 20 °C | - 기압 ≈ 1026 hPa, 습도 ≈ 53–61 % |

> **핵심 요약**  
> 1) 대부분의 선박이 정박·대기 구간과 고속 주행을 번갈아 가며 운항했으며, 급속한 선회가 반복된 단계를 포함합니다.  
> 2) 동일한 날씨 조건(남서풍 10 kn, 파고 0.5 m 이하)에서 운항했으나, GUST가 8 ~ 9 m / s 정도로 선박 안정성에 영향을 미칠 수 있는 표면낙수/전력 변동이 관찰됩니다.

------------------------------------------------------------------
## 2. 개별 배별 특이사항 (숫자 그대로)

| 선박 | 대표 특이점 |
|------|------------|
| **ship_0** | 15:07–18:05 동안 “우선회‧좌선회” 반복으로 회전 반경이 급 급 좁아지고, 18:10–18:18에 48–80° 급선회가 연속. |
| **ship_1** | 5–6 시 비정상적 대형 (≈100° ~ 150°) 회전과 가속/통과가 빈번, 8:30–12:18 동안 3 h 30 min의 “정박·대기”이며 이 후 12 h 가량 고속(≈ 12–13 kn) 직진. |
| **ship_2** | 01:05, 02:45, 03:15, 08:00, 12:00 등에서 20‑30° 정기회전과 0–1 kt 정박이 반복, 여러 시점에서 “정박·대기”가 장시간 지속. |
| **ship_3** | 15:00–15:55에서 급가속(10.5 ~ 11.9 kn)과 급회전(52°‧66°‧23°‧6°)이 한 번에 발생. |
| **ship_4** | 3:05–3:19에 50–90° 우선회 2회, 7:00–8:30 동안 “정박·대기”가 0.9–1.3 kn이면서 여러 작은 회전. |
| **ship_5** | 11:30–12:57에 1 h 30 min 장시간 정박, 14:56–15:34에서는 6‑8 kn 가속과 좌선회가 반복. |
| **ship_6** | 3~5 시 북서‑북동 경로에서 63° 이하의 고속(≈ 13–15 kn) 가속·감속 회전이 빈번, 5 시 이후 빠른 북서 장피로 전환. |
| **ship_7** | 01:12–01:50에서 83°~35° 대회 부호와 16 kn 이상 고속이 같이 나타남, 11:05–13:15 사이에 2 h 10 min 정박. |
| **ship_8** | 01:37–04:38 동안 7.94 kn 평균 속도, 09:04–09:22 0.53 kn 정박, 11:42–14:35에서 14.51 kn 평균 속도. |
| **ship_9** | 08:11–10:49에서 21–23 kn 고속 운항 중 30° ~ 115° 대회 회전이 연속, 15:45–19:49에서 20–23 kn 빠른 복귀선(≈ 150°–170°). |

> **요약**  
> - **고속 회전**(> 30°)과 **갑작스러운 가속**이 여러 선박에서 반복, 특히 ship_3과 ship_7에서 급가속과 급회전이 한 번에 일어남.  
> - **장시간 정박·대기**(≥ 3 h) 구간은 ship_1, ship_4, ship_8 등에서 흔하게 나타남, 이는 선박 이동 경로와 항구 접근이 연관될 가능성이 큼.  
> - **캣버🡒전선 변동**(예: ship_6 북서→북동, ship_2 북동→남동 등)이 풍향(337°)과 매칭돼 선체 안정성에 관여할 수 있음.

------------------------------------------------------------------
## 3. 권고 사항 (공통 & 특이)

| 항목 | 공통권고 | 소스 |
|------|----------|------|
| **선회 시 속도** | 10 kn 이하 유지, 급회전 시 회전 반경 확대 | 모든 선박 |
| **정박 시** | 풍향이 대항면에 대조되면 등록‑역풍 방지, 선체 균형감시 | 모든 선박 |
| **GUST 반응** | 8 ~ 9 m / s 돌풍 시 속도 60 % 이하로 임시 감속 | 모든 선박 |
| **높은 파고(> 1 m)** | 고속+회전 시 10 kn 이하로 낮춤, 파고 변동 집중 감시 | ship_0, ship_4 등에서 관찰 |
| **특이 속도 변화** | ship_3, ship_7 15:00‑15:55, 01:12‑01:50 등 급가속+고속 시 경고 | ship_3/7 |
| **정박·대기 기간 > 3 h** | 선박 간 거리를 500 m 이상 확보, AIS 재확인 | ship_1, ship_4, ship_8 등 |

> **결론**  
> 모든 선박이 동일한 남서풍(≈10 kn)과 낮은 파고·GUST 조건에서 운항했음에도, 선회와 가속이 빈번한 구간은 해상 충돌, 선체 흔들림, 조정 지연 위험이 존재합니다. 따라서 각 선박의 특이 시각(특히 급가속-급회전 구간 및 장시간 정박 시)마다 별도 경고 및 속도 조절이 필요합니다.

In [126]:
#병렬로 처리
analyze_chain = each_ship_prompt | local_llm | StrOutputParser()

# --- 1. 개별 분석 함수 (스레드에서 실행될 단위) ---
def analyze_single_ship(data):
    # data는 {'idx': i, 'trajectory': t, 'weather': w} 형태
    res = analyze_chain.invoke({
        "trajectory_data": data['trajectory'],
        "weather_data": data['weather']
    })
    return f"[ship_{data['idx']+1} 분석 결과]\n{res}"

# --- 2. 병렬 처리를 수행하는 래퍼 함수 ---
def parallel_wrapper(inputs):
    # inputs: {'tracks': [...], 'weathers': [...]}
    tracks = inputs['tracks']
    weathers = inputs['weathers']
    
    tasks = [
        {'idx': i, 'trajectory': tracks[i], 'weather': weathers[i]}
        for i in range(len(tracks))
    ]
    
    with ThreadPoolExecutor(max_workers=100) as executor:
        results_list = list(executor.map(analyze_single_ship, tasks))
    
    # 다음 체인(요약)을 위해 딕셔너리 형태로 반환
    return {
        "all_analyses": "\n\n".join(results_list),
        "ship_count": len(tracks)
    }

# --- 3. 하나의 거대한 체인으로 결합 ---
full_integrated_chain = (
    RunnableLambda(parallel_wrapper)  # 1단계: 병렬 분석 실행 및 결과 취합
    | summary_trajectory_prompt       # 2단계: 요약 프롬프트에 전달
    | local_llm                       # 3단계: 최종 요약 생성
    | StrOutputParser()               # 4단계: 텍스트만 추출
)

In [131]:
print(full_integrated_chain)

first=RunnableLambda(parallel_wrapper) middle=[ChatPromptTemplate(input_variables=['all_analyses'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['all_analyses'], input_types={}, partial_variables={}, template='\n    #Input Data\n    {all_analyses}\n\n    # Guidelines\n    1. Input Data의 정보를 종합해서 전체 배에 대한 내용을 종합적으로 요약해주세요\n    2. 요약 시 각각의 배의 이름을 명시하고 특이사항을 중점적으로 요약하세요 \n    3. 수치는 바뀌지 않도록 정확하게 참고해주세요\n    4. 반복적인 내용은 짧게 요약하세요\n    '), additional_kwargs={})]), ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001974BF01D10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001974BF020D0>, root_client=<openai.OpenAI object at 0x000001974BF01A90>, root_async_client=<openai.AsyncOpenAI object at 0x000001974BF01E50>, model_name='openai/gpt-oss-20b', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://192.168.0.1

In [40]:
# --- 4. 실행 ---
start = time.time()
parallel_result = full_integrated_chain.invoke({
    "tracks": ship_track_list,
    "weathers": weather_results
})

print(time.time() - start)

95.87451767921448


In [41]:
display(Markdown(parallel_result))

**전반적 상황 개요**  
1. 모든 선박은 12월 1일 00:00 ~ 23:59 시간대에 Ulsan(35.35° N, 129.84° E)에서 관측된 동일한 풍속·풍향(≈ 5.9 m s⁻¹ ~ 7.6 m s⁻¹, 337° ~ 23°)과 0.6 ~ 0.9 m 의 파고 아래 운항했습니다.  
2. 정박·부트·가속·감속을 반복하며 대부분 6 ~ 15 kts (≈ 10 ~ 27 kn) 사이에서 주행했으며, 대다수 선박에서 “급진적(> 30°) 좌우 선회”가 빈번했습니다.  
3. 장시간 정박(> 6 h) 구간이 다수 존재했으며, 해당 구간에서 드래프트·풍압·파고 변화를 주의해 정박 안전성을 점검해야 함.  
4. 풍향이 북‑서(337°)에서 남‑서(23°)까지 변동해 소위 “역풍”·“측풍”이 발생했고, 이를 대비해 가속·감속 시 바람·파동 반응을 스스로 조정하도록 권고합니다.

---

## 각 선박별 핵심 특이사항

| 선박 | 중요 운항 패턴 | 핵심 이벤트 | 주의 포인트 |
|---|---|---|---|
| **AOE‑59 화천** | 06:06(정박) → 08:32(좌선) → 14:55(우선) → 18:10(반시계) → 20:35(좌선) → 21:59(정박 종료) | 8 ~ 15 kts 좌우 선회 태세; 08:32‑14:50 집중 정박, 18:10‑20:35 우선·좌선 교대로 10–12 kts | 강풍·파고 대비 30° 선회 시 선체 드리프트 주의 |
| **ATS‑평택** | 00:00(저속) → 05:45(정박) → 07:00(가속) → 08:00‑09:00 정박/좌선 | 10 ~ 12 kts 대규모 좌우 선회(≥ 100°)와 장시간 정박 | 정박 시 드래프트·풍압 확인, 선체 기울기 모니터링 |
| **FF‑959 부산** | 00:00-24:00 거대한 변선·저속 주기(0.1 ~ 10 kn) | 20–30° 급선회·정박사이로 수십 회 반복 | 매우 짧은 거리에서의 빈번한 선회 → 충돌 위험 |
| **FFG‑818 대구** | 14:21(정박) → 15:06(우선·좌선) → 16:06(가속) → 18:52(저속) | 14:21 이후 10–12 kts 가속으로 90°+ 기동 | 5.5 m s⁻¹(W)-9.1 m s⁻¹(G) 바람·파고 조합 시 정박 전개 불안정 |
| **LST‑675 개봉** | 07:00(정박) → 09:10(가속 10 ~ 11 kn) → 22:31(정박) → 23:54(마지막 탐지) | 08 ~ 11 kn 지속 이동; 22 ~ 23 h 간 국제국가에 근접 | 12 kn 거친 바람·1.6 m 파고 대비 속도 감소(≤ 8 kn) |
| **VTK‑공역** | 00:06‑11:30(10.7 kn) → 11:42‑15:22(≤ 3 kn) → 15:26‑16:49(5‑8 kn) | 10 ~ 15 kn 가속·감속, 좌우 선회(20‑40°) | 8 ~ 17 kn 구간에서 풍압·파고 충돌 가능성 |
| **LST‑683 향로봉** | 00:00‑04:50(12 ~ 15 kn) → 04:55‑05:12(기울기 고정) | 40‑60° 급선회 반복 | 5.9 m s⁻¹(북‑서) 바람에 민감, 선체 드리프트 주의 |
| **SS‑062 이천** | 00:00‑01:10(정박) → 01:12‑06:26(9‑16 kn) → 11:05‑22:47(0.4 ~ 17 kn) | 83° 좌선·35° 우선(고속) | 16‑17 kn 구간에서 5.9 m s⁻¹/23° 풍향에 따른 헤드스탠스 상승 |
| **SS‑075 안중근** | 01:37‑04:38(7.9 kn) → 07:04‑09:22(정박/좌선) → 11:12‑14:35(14.5 kn) → 18:39‑21:24(11.5 kn) | 80°+ 우선·좌선 반복, 4 ~ 15 kn 변동 | 4.4 m s⁻¹(북‑남) 바람과 파고 0.5 m 이하, 급풍(6 m s⁻¹) 주의 |
| **SS‑078 유관순** | 04:03‑08:03(정박) → 08:11‑10:49(17 kn) → 11:04‑15:40(우선·정박) → 22:37‑23:14(20‑22 kn) | 50°+ 선회·가속·감속 연속 | 7 m s⁻¹(NE) 풍향 29°, 파고 0.3‑0.6 m; 고속 구간에서 흔들림 면밀히 |

> **공통 주의점**  
> * 여러 선박에 걸친 **30°‑80°** 크기의 급선회는 항해 경로에서 좁은 영역을 통과할 가능성이 크므로 비상 회피 또는 선박간 거리 유지에 명시적 확인이 필요합니다.  
> * **산정에 따른 바람·파고로 인한 드리프트**는 정박 시 특히 위험하며, 핵심 선박(AOE‑59, ATS‑평택, FFG‑818, VTK‑공역)은 정박 중 드래프트 및 풍압 변동을 세심히 감독합니다.  
> * 정규 노선에서 **풍향**이 북‑서(337°)에서 남‑서(23°)로 변동하게 되면 풍향·파향이 서로 다른 방향으로 작용, 선체가 “측풍”으로 부러지는 경우가 발생하므로, 선박 전진률과 헤드스탠스를 조정하세요.  
> * 전체 선박이 경험한 **간헐적 풍속 급등(8‑9 m s⁻¹)** 은 갑작스러운 가속·감속을 유발하므로, **AIS**·**VTS** 간 신속한 데이터 공유가 필수입니다.  

위 표와 요약을 토대로 VTS 팀은 **각 선박별** 접근 시점(정박/선회/가속 구간)을 신속히 조정하고, **운항 경로**와 **날씨 변수**를 매니지먼트 하여 사고 방지·운항 효율을 극대화해야 합니다.

In [49]:
#astream 첫 chunk가 출력되는 시간 계산 함수
async def astream_output():
    full_text = ""
    handle = display(Markdown(""), display_id=True)
    
    start_time = time.time()
    ttft = None
    # astream을 사용하여 비동기 스트리밍
    async for chunk in full_integrated_chain.astream({
        "tracks": ship_track_list,
        "weathers": weather_results
    }):
        if ttft is None:
            ttft = time.time() - start_time
            print("---시간계산---")
            print(f"{ttft}초")
        # 주피터 노트북에서 바로 출력 (end=""로 줄바꿈 없이 출력)
        full_text += chunk
        handle.update(Markdown(full_text))

In [50]:
# 주피터에서 비동기 함수 실행
await astream_output()

**전체 요약**  
12 월 1일 12 시 기준, 10척대(ship_1~ship_10) 모두 동해·남해 사근해역에서 운항 중이며, 대부분 북‑북서풍 5.5–6.0 m s⁻¹(≈10–12 kn)과 0.2–1.5 m 사이의 파고가 적용됐다.  
단, 각 선박은 개별적인 선회(좌/우 >50°), 가속(10–18 kn), 고속 주행(≈20 kn) 특성을 보이며, “정박·대기”가 장시간 지속되는 구간이 다수 존재한다.  
관제는 **풍속·풍향으로 인한 선회·속도 변동**과 **파고 영향**을 중심으로, **주요 선회 구간(15:30–18:06, 20:25–22:00 등)**에서 가시성 향상 및 경로 검토를 권고하고, **정박 구간**에서 선박 점검 및 충돌‑안전 보장을 강조한다.  

---

### 1. ship_1  
- **주요 항해 특징**: 15:30–18:06, 20:25–22:00 로 11 – 12 km h⁻¹ 고속와 복합 선회 반복.  
- **정박·대기**: 06:06–14:50이 장기간(≈8 h) 고속히 잡아두고, 08:32–08:35 짧은 좌선회(98.3°)와 14:55–15:30 우선회(88.4°) 진행.  
- **안전 포인트**: 고속 구간의 파고 0.9 m, 북서풍 5.9 m s⁻¹. 선박가속 시 바람·파고 조합에 따라 선반정 속도 변동 위험.

### 2. ship_2  
- **운항 패턴**: 00:00–05:40 저속 1.3 kn, 05:51–06:00 가속 7–11 kn, 08:48–12:18 정박·대기(0.33 kn) 3 h 36 분, 12:22–14:16 고속 12–13 kn.  
- **특이 사항**: 06:05–06:11 7 → 11 kn 가속, 03, 04, 17, 19, 21, 22, 28 단계에서 50° > 선회, 12:48–12:18 일정 정박, 19 단계 -144° 대대적 무빙.  
- **관제 주의**: 잦은 선회·가속으로 인한 충돌 위험, 정박 구간에서 조력·안전 점검 필요.

### 3. ship_3  
- **항해 흐름**: 00:00–01:00 6–10 kn, 01:05–03:45 0–1 kn 정박 대기, 04:00–07:35 0–2 kn, 08:00–12:00 1.5–2 kn, 13–15 kn, 15:06–18:30 2 kn, 18:00–22:00 1.3 kn.  
- **특이**: 06:35–07:35 0–1 kn, 15 ~ 15:55 11.9 kn, 22:25–23:55 5–11 kn.  
- **관제 포인트**: 지속적 정박·저속 구간(03:50–07:35)에서 충돌 방지, 고속 구간 파고 0.9 m 대비 선회 관리.

### 4. ship_4  
- **운항 패턴**: 00:00–01:40 저속·정박, 07:35–11:00 1–2 kn, 11:05–22:00 주로 1–2 kn, 22:25–23:55 5–11 kn 가속.  
- **특이사항**: 가속 이동/통과 14:47–15:06 10–11 kn, 좌선회 3–4 분 0–1 h.  
- **관제**: 가속·좌선회 시 방향 변동 주시, 정박 시 부두·방해물 거리를 재확인.

### 5. ship_5  
- **항해 흐름**: 00:00–03:04 1.6 kn, 06:40–07:00 4 kn, 09:15–11:19 9.5 kn, 11:25–22:31 10.8 kn, 22:47–23:04 11.1 kn.  
- **특이사항**: 09:10–09:14 가속 6.8 kn, 07–08 사이 저속·정박.  
- **관제**: 09:10~11:19 높음가속 구간에서 충돌 위험, 07–08 파도 1.6 m, 6.7 m s⁻¹ gust 대비 가속 조정 필요.

### 6. ship_6  
- **정박·대기**: 00:00–01:09 0.04 kn, 01:12 가속 2.4 kn, 01:18–06:26 16 kn(≈30 kt) 고속, 06:30–10:55 11–17 kn, 11:05–13:16 1.5 h 정박.  
- **특이**: 01:18–02:00, 06:30–07:32 고속, 11‑13 정박 후 2–6 kn 재출항.  
- **관제 포인트**: 고속 회전 (30–35°) 시 피로·충돌 위험, 정박 시 물류 차선 영향 완화.

### 7. ship_7  
- **항해 흐름**: 00:00 ~ 05:50 고속 (≈12–15 kn) 및 다중 40° ~ 50° 회전.  
- **특이**: 02:00–03:00 가속/감속(13–15 kn), 04:05–04:25 좌선회(±64°) 반복.  
- **관제**: 다중 회전 구간(00:30–1:55, 04:05–4:35)에서 충돌 방지, 파고 1.5 m 대비 선회력 관리.

### 8. ship_8  
- **운항 패턴**: 00:00–01:09 정박, 01:12 2.4 kn 가속, 01:18–06:26 고속 16 kn, 06:30–10:55 11–17 kn, 11:05–13:16 1.5 h 정박, 13:26–22:47 15–17 kn 고속.  
- **특이사항**: 01:18~02:00, 06:30~07:32 고속 회전; 11–13 정박하여 물류 차선 곤란.  
- **관제 포인트**: 고속 지연 회전으로 인한 선체 피로, 정박 시 접근선 관리 필요.

### 9. ship_9  
- **항해 흐름**: 01:37–04:38 7.94 kn, 07:04–09:22 정박, 09:26–10:57 1.0–1.25 kn, 11:12–11:31 6–14 kn 가속, 11:42–14:35 14.5 kn, 14:50–18:14 14.8–18 kn, 18:39–21:24 11.5 kn 감속, 21:25–23:57 정박 (71° 회전).  
- **특이**: 23:20~23:57 71° 회전, 20:39 ~ 21:24 감속 11–12 kn.  
- **관제**: 정박·대기 구간에서 충돌 방지, 09:26–10:57 저속 구간에서 파고·풍향(북‑북동 4°) 대비 선회.

### 10. ship_10  
- **항해 흐름**: 04:03–08:03 정박, 08:11 14.4 kn (우선회 26°), 08:26–10:49 21 kn, 11:04–15:40 정박, 15:45–17:24 21 kn, 17:27–19:49 21 kn, 20:04–20:19 감속 (16→4.8 kn), 22:37–23:14 21 kn (103° 급좌선회).  
- **특이 사항**: 08:26–10:49 고속 주행, 20:04–20:19 감속, 22:37 급좌선회.  
- **관제**: 고속 회전(≥50°) 시 선박 안정성 체크 및 정박 구간에서 바람·파고(0.4 m) 대비 선반정 회전 관리.

---

**공통적인 관제 지침**

| 구간 | 주의 포인트 | 권고 조치 |
|------|--------------|-----------|
| **고속 가속/좌우 선회** (10–20 kn) | 선회각 50°–70° 시 선일정·바람·파고 대결 | AIS·VTS 실시간 모니터링, 선원 주의 및 속도 제한 |
| **장기 정박·대기(0~1 kn)** | 선박 충돌 가능성, 파고·풍향 인과원거리 멀기 | 주변 선박 거리 150–200 m 확보, 정박선박 점검(방수·체인) |
| **풍향 337° (북서풍), 5.5–6 m s⁻¹** | 반대 방향 바람에 의한 리액션 교정 | 회전 전후 속도 조정, 선박 회전율 제한 |
| **파고 0.2–1.5 m** | 고속 운항 시 부드러운 흔들림에서 선체 동작 영향 | 고속 시 선체 주파수·롤 동기화, 가속 제한 |

이상의 정보를 토대로 각 선박별 항로 확인, 선회·가속·정박 시점에서의 실시간 감시·명령 전송을 강화하여 해상 안전을 최우선으로 하여 항해를 관리하시기 바랍니다.

61.73719358444214초


In [45]:
full_text = "".join(result_lst)
display(Markdown(full_text))

## 전체 배계정 요약  
- 10 척의 선박은 전면 (울산‑거제‑총신‑통영‑우해) KERI 동해 해역에서 2022‑12‑01 0 시‑23 시 동안 운항.  
- 풍향 337° (북서) ≈ 10 kn, 파고 ≤ 1.5 m; 온도 ≈ 9 °C, 습도 ≈ 53 % (≈ 가벼운 한파).  
- 대부분 **저속(≤ 3 kn)** → 짧은 정박이 두드러지며, **고속(≈ 15–21 kn)** 로는 **급격 회전**(50° ~ 150°)이 빈번.  
- 모두 급격 가속(≥ 7 kts) 또는 **초과회전(≥ 90°)**을 보였으며, 특히 **장시간 정박(> 3 h)** 이 존재, 조기 충돌 및 서스펜션 이슈 리스크 증가.  
- 모두 **풍속 ≈ 10 kn** 직전이나 18 km h 이상 운항 시 파고 ≤ 1.5 m 이므로 단계적 안전 관리에 집중.  

---

## 개별 선박 요약

| # | 선박명(원문) | 주요 운항 특이사항 | 관제 집중 포인트 |
|---|--------------|------------------|------------------|
| 1 | **AOE‑59 화천** | 06:06‑14:55  장시간 정박. 15 시에 급속 가속(53° 우선회 10.9 kn) → 21 시까지 10–12 kn. 14–15 시 큰 초과회전(−164.7°, −98.3°). | **번역**: 정박시간 초과, 초과회전 시 선체 탄성 검출 필요. |
| 2 | **[ship_2]** | 00:00‑07 시끼리 **수 저속·정박(3 시)**. 14–15 시 고속(≈ 13 kn)→**긴 장거리**(≈ 10 h). | **무인 선박?**: 정박·우선회 빈번 → 충돌 방지 라인 확보. |
| 3 | **[ship_3]** | 00–01 시 정박, 01:05–02:15 **저속**(≤ 2 kn). 22–23 시~**고속**(≈ 10 kn). | **주제**: 저속에서 고속으로 전환 시 바람·파고 낮아도 **접근 경로** 주의. |
| 4 | **[ship_4]** | 여러 가속·회전(90–160°)→**대사적으로 11–14 시** 고속(≈ 10 kn) → 15–16 시 **저속**, 23 시 고속(≈ 11 kn). | **장황 정박**(01:35–04:45); **극단적 회전**(14‑15 시). |
| 5 | **[ship_5]** | 03:40–05 시 정박 → 09 시부터 **고속(≈ 10 kn)** 진행(≈ 13 h). 22 시 좌선회·가속→**전환**. | **긴 고속기동** (09–22 시) → 풍향 가온대 선율 보호. |
| 6 | **[ship_6]** | 00:06 시 출항→10‑20 km/h 고속, **13 시–15 시** 가속·감속 → 16 시 정박(6 h). | **정박·저속** 이후 **가속(15–18 시)** → 바람풍향이 변경될 때 신속 대응. |
| 7 | **[ship_7]** | 00:00 ~ 05 시 **연속 움직임** 12–15 kn → 미세 회전(42–64°). | **좌우빈번 회전** → 조향·가속 0.5–1 % 오차 보정 필각. |
| 8 | **[ship_8]** | 00 ~ 01 시 정박 → 01:19 83° 급좌선회 → 고속(≈ 16 kn) → 07 시 저속(≤ 3 kn). | **초극적인 좌선회(83°)** → 선체 매듭 감시. |
| 9 | **[ship_9]** | 01 ~ 07 시 정박·저속→11 시 가속(≥ 14 kn)→14 시 고속(≈ 15 kn). 23 시가 가까워서 정박 → 점차 저속. | **낮은 바다 온도** (19 °C) → 가시성·체온 대책 필요. |
|10 | **[ship_10]** | 04 시 정박→08 시 고속(≈ 21 kn), 11 시 좌우 거대 회전, 15 시 가속→19 시 주행, 20 시 저속(≤ 10 kn) → 23 시 고속(≈ 21 kn). | **센트럴링(수정선둥)** → 풍향과 반대 이동 경로 유지.

> **주의**: 모든 선박이  군상적 **회전(> 90°)** 또는 **초과 회전**을 다수 보였고, 대부분의 **정박시간이 3 h 초과** 되어 출입국관제와 충돌 위험이 조기 사전 주의 필요.

---

## 관제 조언  
1. **정박상황**: 3 시 이상 정박 시 선체 진동, 윤활, 손상 가능성 감시.  
2. **급 회전**: 50–150° 회전이 급속(≥ 7 kts)과 결합될 때, **레버와 선체 진동**에 주의.  
3. **풍속·풍향**: 10 kn의 북서풍이 대부분의 고속 구간에 반대이며, **풍향과 고속 경로**가 180°이상 차이 나면 서브제트 현상 발생 가능.  
4. **파고**: 1.5 m 이하 일상, 고속 운항 시 **파향 23°**와 반대 방향으로 선박이 기울어짐(좌우 가속).  
5. **시간표**: 모든 선박이 0–24 시 불연속 12 h 이상 고속을 채택 → **흘림** 및 **장거리 회전**에 대한 정기 점검 권고.  

위 10 척 모두 **기상·항로 조정**과 **정박 · 회전**을 동시 모니터링하여 안전한 운영을 수행할 수 있도록 하오.